# Phone Product GraphRAG — Neo4j Import Pipeline

Pipeline này import dữ liệu điện thoại từ CSV vào Neo4j theo kiến trúc GraphRAG đã thiết kế:

```
Brand → Series → Model → ModelStorage (giá)
                       → Variant (SKU) → ColorVariant → ColorFamily
                                       → ColorImage
                       → SpecCategory → SpecItem (ống kính)
                                      → (features[] as property)
```

**Yêu cầu:** Neo4j >= 5.x, Python 3.9+, `pip install neo4j pandas`

## 0. Cài đặt thư viện

In [ ]:
# !pip install neo4j pandas

## 1. Import & Config

In [9]:
import json
import re
import pandas as pd
from neo4j import GraphDatabase
from typing import Optional

# ─── Neo4j connection ─────────────────────────────────────────
NEO4J_URI      = "bolt://172.18.224.1:7687"
NEO4J_USER     = "neo4j"
NEO4J_PASSWORD = "12345678"

# ─── Data source ──────────────────────────────────────────────
CSV_PATH = "crawl_data/data/test_data.csv"

# ─── Brand config (hardcoded vì data là Apple) ────────────────
BRAND_NAME = "Apple"

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print("✅ Neo4j connected")

✅ Neo4j connected


## 2. Helpers: Parser & Normalizer

In [ ]:


# ─── Price string → float ─────────────────────────────────────
def parse_price(price_str: str) -> Optional[float]:
    """Convert '36.990.000đ' hoặc '36.990.000₫' → 36990000.0"""
    if not price_str or pd.isna(price_str):
        return None
    cleaned = re.sub(r'[^\d]', '', str(price_str))
    return float(cleaned) if cleaned else None


# ─── Extract model info từ product name ───────────────────────
def parse_model_info(name: str, sku: str) -> dict:
    """
    Từ 'iPhone 17 Pro Max 256GB | Chính hãng' và sku 'iphone-17-pro-max'
    → {'series': 'iPhone 17', 'model': 'iPhone 17 Pro Max',
       'storage_gb': 256, 'tier': 'pro_max'}
    """
    # Bóc storage
    storage_match = re.search(r'(\d+)\s*(GB|TB)', name, re.IGNORECASE)
    storage_gb = None
    if storage_match:
        val  = int(storage_match.group(1))
        unit = storage_match.group(2).upper()
        storage_gb = val * 1024 if unit == 'TB' else val

    # Bóc tier
    name_lower = name.lower()
    if 'pro max' in name_lower:
        tier = 'pro_max'
    elif ' pro' in name_lower:
        tier = 'pro'
    elif ' plus' in name_lower:
        tier = 'plus'
    elif ' air' in name_lower:
        tier = 'air'
    elif 'iphone 16e' in name_lower or 'iphone 17e' in name_lower or sku.endswith('e'):
        tier = 'e'
    else:
        tier = 'standard'

    # Bóc model name (bỏ storage + phần sau |)
    model_name = name.split('|')[0].strip()
    model_name = re.sub(r'\s+\d+(GB|TB).*$', '', model_name, flags=re.IGNORECASE).strip()

    # Bóc series (model bỏ tier suffix)
    series_name = model_name
    for suffix in [' Pro Max', ' Pro', ' Plus', ' Air', 'e']:
        if series_name.endswith(suffix):
            series_name = series_name[:-len(suffix)].strip()
            break

    # Bóc release year từ model name (vd iPhone 17 → 2025, iPhone 16 → 2024)
    gen_match = re.search(r'iPhone (\d+)', model_name)
    gen_num   = int(gen_match.group(1)) if gen_match else None
    year_map  = {14: 2022, 15: 2023, 16: 2024, 17: 2025}
    release_year = year_map.get(gen_num)

    return {
        'series_name' : series_name,
        'model_name'  : model_name,
        'tier'        : tier,
        'storage_gb'  : storage_gb,
        'release_year': release_year,
    }


# ─── SpecCategory key normalizer ──────────────────────────────
SPEC_CATEGORY_MAP = {
    'Màn hình'              : 'man-hinh',
    'Camera sau'            : 'camera-sau',
    'Camera trước'          : 'camera-truoc',
    'Vi xử lý & đồ họa'    : 'chip',
    'Bộ xử lý & Đồ họa'    : 'chip',
    'Giao tiếp & kết nối'  : 'ket-noi',
    'RAM & lưu trữ'         : 'ram-luu-tru',
    'Pin & công nghệ sạc'   : 'pin',
    'Kích thước & Trọng lượng': 'thiet-ke',
    'Thiết kế & Trọng lượng': 'thiet-ke',
    'Thông số khác'         : 'thong-so-khac',
    'Tiện ích khác'         : 'tien-ich',
    'Tính năng khác'        : 'tinh-nang-khac',
    'Cổng kết nối'          : 'cong-ket-noi',
    'Thông tin chung'       : 'thong-tin-chung',
}

# ─── SpecItem → tách ống kính camera thành node riêng ─────────
LENS_SPEC_NAMES = {
    'Camera sau', 'Camera chính', 'Telephoto', 'Góc siêu rộng',
    'Camera trước', 'Quay video', 'Quay video trước',
}

def is_lens_spec(name: str) -> bool:
    """Ống kính → node SpecItem riêng. Các thứ khác → property."""
    return any(k.lower() in name.lower() for k in LENS_SPEC_NAMES)


# ─── Extract numeric value từ spec value string ────────────────
def extract_numeric(value_str: str) -> Optional[float]:
    match = re.search(r'([\d]+(?:[.,][\d]+)?)', str(value_str))
    if match:
        return float(match.group(1).replace(',', '.'))
    return None


print("✅ Helpers ready")

✅ Helpers ready


## 3. Load & Preview Data

In [12]:
CSV_PATH = "data/test_data.csv"
df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} products")
print()

# Parse model info cho từng row
df['_model_info'] = df.apply(
    lambda r: parse_model_info(r['name'], r['sku']), axis=1
)

# Preview
preview = df.apply(lambda r: {
    'sku'        : r['sku'],
    'model'      : r['_model_info']['model_name'],
    'series'     : r['_model_info']['series_name'],
    'tier'       : r['_model_info']['tier'],
    'storage_gb' : r['_model_info']['storage_gb'],
    'base_price' : parse_price(r['base_price']),
    'sale_price' : parse_price(r['sale_price']),
}, axis=1)

pd.DataFrame(preview.tolist())

Loaded 10 products



,sku,model,series,tier,storage_gb,base_price,sale_price
0,iphone-17-pro-max,iPhone 17 Pro Max,iPhone 17,pro_max,256,37990000.0,36990000.0
1,iphone-17-pro,iPhone 17 Pro,iPhone 17,pro,256,34990000.0,34690000.0
2,iphone-17-256gb,iPhone 17,iPhone 17,standard,256,24990000.0,24390000.0
3,iphone-16-pro-max,Điện thoại iPhone 16 Pro Max,Điện thoại iPhone 16,pro_max,256,34990000.0,30990000.0
4,iphone-17e,iPhone 17e,iPhone 17,e,256,17990000.0,17490000.0
5,iphone-15,iPhone 15,iPhone 15,standard,128,19990000.0,17590000.0
6,iphone-16-pro-max-512gb,iPhone 16 Pro Max,iPhone 16,pro_max,512,40990000.0,38490000.0
7,iphone-17-pro-max-512gb,iPhone 17 Pro Max,iPhone 17,pro_max,512,44490000.0,42990000.0
8,iphone-14,iPhone 14,iPhone 14,standard,128,14990000.0,13990000.0
9,iphone-16e,iPhone 16e,iPhone 16,e,128,16990000.0,11990000.0


## 4. Schema: Tạo Constraints & Indexes

In [13]:
CONSTRAINTS = [
    # Unique constraints (cũng tự tạo index)
    "CREATE CONSTRAINT brand_name IF NOT EXISTS FOR (b:Brand) REQUIRE b.brand_id IS UNIQUE",
    "CREATE CONSTRAINT series_id IF NOT EXISTS FOR (s:Series) REQUIRE s.series_id IS UNIQUE",
    "CREATE CONSTRAINT model_id IF NOT EXISTS FOR (m:Model) REQUIRE m.model_id IS UNIQUE",
    "CREATE CONSTRAINT model_storage_id IF NOT EXISTS FOR (ms:ModelStorage) REQUIRE ms.model_storage_id IS UNIQUE",
    "CREATE CONSTRAINT variant_sku IF NOT EXISTS FOR (v:Variant) REQUIRE v.sku_id IS UNIQUE",
    "CREATE CONSTRAINT color_variant_id IF NOT EXISTS FOR (cv:ColorVariant) REQUIRE cv.color_variant_id IS UNIQUE",
    "CREATE CONSTRAINT color_family_id IF NOT EXISTS FOR (cf:ColorFamily) REQUIRE cf.family_id IS UNIQUE",
    "CREATE CONSTRAINT spec_cat_id IF NOT EXISTS FOR (sc:SpecCategory) REQUIRE sc.spec_category_id IS UNIQUE",
    "CREATE CONSTRAINT spec_item_id IF NOT EXISTS FOR (si:SpecItem) REQUIRE si.spec_item_id IS UNIQUE",
    # Indexes cho query nhanh
    "CREATE INDEX model_chip IF NOT EXISTS FOR (m:Model) ON (m.chip)",
    "CREATE INDEX model_display IF NOT EXISTS FOR (m:Model) ON (m.display_size_in)",
    "CREATE INDEX spec_item_name IF NOT EXISTS FOR (si:SpecItem) ON (si.name)",
]

with driver.session() as session:
    for cypher in CONSTRAINTS:
        session.run(cypher)
        print(f"  OK: {cypher[:60]}...")

print("\n✅ Schema created")

  OK: CREATE CONSTRAINT brand_name IF NOT EXISTS FOR (b:Brand) REQ...
  OK: CREATE CONSTRAINT series_id IF NOT EXISTS FOR (s:Series) REQ...
  OK: CREATE CONSTRAINT model_id IF NOT EXISTS FOR (m:Model) REQUI...
  OK: CREATE CONSTRAINT model_storage_id IF NOT EXISTS FOR (ms:Mod...
  OK: CREATE CONSTRAINT variant_sku IF NOT EXISTS FOR (v:Variant) ...
  OK: CREATE CONSTRAINT color_variant_id IF NOT EXISTS FOR (cv:Col...
  OK: CREATE CONSTRAINT color_family_id IF NOT EXISTS FOR (cf:Colo...
  OK: CREATE CONSTRAINT spec_cat_id IF NOT EXISTS FOR (sc:SpecCate...
  OK: CREATE CONSTRAINT spec_item_id IF NOT EXISTS FOR (si:SpecIte...
  OK: CREATE INDEX model_chip IF NOT EXISTS FOR (m:Model) ON (m.ch...
  OK: CREATE INDEX model_display IF NOT EXISTS FOR (m:Model) ON (m...
  OK: CREATE INDEX spec_item_name IF NOT EXISTS FOR (si:SpecItem) ...

✅ Schema created


## 5. Import Functions

In [14]:
# ─── 5.1 Brand & Series ───────────────────────────────────────

def upsert_brand_series(tx, brand_name: str, series_info: dict):
    tx.run("""
        MERGE (b:Brand {brand_id: $brand_id})
        ON CREATE SET b.name = $brand_name, b.country = 'USA'

        MERGE (s:Series {series_id: $series_id})
        ON CREATE SET s.name = $series_name

        MERGE (b)-[:HAS_SERIES]->(s)
    """, {
        'brand_id'   : brand_name.lower(),
        'brand_name' : brand_name,
        'series_id'  : series_info['series_id'],
        'series_name': series_info['series_name'],
    })


# ─── 5.2 Model ────────────────────────────────────────────────

def upsert_model(tx, model_data: dict):
    """
    model_data phải có: model_id, name, series_id, tier, release_year,
    summary, tagline, colors_available, storage_options,
    + denormalized specs (chip, display_size_in, ...)
    """
    tx.run("""
        MATCH (s:Series {series_id: $series_id})

        MERGE (m:Model {model_id: $model_id})
        ON CREATE SET
            m.name             = $name,
            m.tier             = $tier,
            m.release_year     = $release_year,
            m.status           = 'available',
            m.summary          = $summary,
            m.tagline          = $tagline,
            m.colors_available = $colors_available,
            m.storage_options  = $storage_options,
            m.chip             = $chip,
            m.display_size_in  = $display_size_in,
            m.display_hz       = $display_hz,
            m.main_camera_mp   = $main_camera_mp,
            m.weight_g         = $weight_g,
            m.ip_rating        = $ip_rating,
            m.back_material    = $back_material,
            m.frame_material   = $frame_material,
            m.os               = $os
        ON MATCH SET
            m.colors_available = $colors_available,
            m.storage_options  = $storage_options

        MERGE (s)-[:HAS_MODEL]->(m)
    """, model_data)


# ─── 5.3 ModelStorage (giá theo storage) ─────────────────────

def upsert_model_storage(tx, model_id: str, storage_gb: int,
                          base_price: float, sale_price: float):
    ms_id = f"{model_id}-{storage_gb}gb"
    tx.run("""
        MATCH (m:Model {model_id: $model_id})
        MERGE (ms:ModelStorage {model_storage_id: $ms_id})
        ON CREATE SET
            ms.storage_gb  = $storage_gb,
            ms.base_price  = $base_price,
            ms.sale_price  = $sale_price,
            ms.currency    = 'VND'
        ON MATCH SET
            ms.base_price  = $base_price,
            ms.sale_price  = $sale_price
        MERGE (m)-[:HAS_CONFIG]->(ms)
    """, {
        'model_id'  : model_id,
        'ms_id'     : ms_id,
        'storage_gb': storage_gb,
        'base_price': base_price,
        'sale_price': sale_price,
    })
    return ms_id


# ─── 5.4 ColorFamily & ColorVariant ──────────────────────────

def upsert_color_variant(tx, model_id: str, color_name: str,
                          image_url: str, color_price: float):
    family_id = re.sub(r'\s+', '-', color_name.lower().strip())
    cv_id     = f"{model_id}-{family_id}"

    tx.run("""
        // ColorFamily (concept màu xuyên đời)
        MERGE (cf:ColorFamily {family_id: $family_id})
        ON CREATE SET cf.name = $color_name

        // ColorVariant (màu cụ thể của model này)
        MERGE (cv:ColorVariant {color_variant_id: $cv_id})
        ON CREATE SET
            cv.name           = $color_name,
            cv.model_specific = $model_id,
            cv.image_url      = $image_url,
            cv.color_price    = $color_price
        ON MATCH SET
            cv.image_url   = $image_url,
            cv.color_price = $color_price

        MERGE (cv)-[:BELONGS_TO_FAMILY]->(cf)
    """, {
        'family_id' : family_id,
        'color_name': color_name,
        'cv_id'     : cv_id,
        'model_id'  : model_id,
        'image_url' : image_url,
        'color_price': color_price,
    })
    return cv_id


# ─── 5.5 Variant (SKU) ────────────────────────────────────────

def upsert_variant(tx, model_id: str, ms_id: str, cv_id: str,
                    storage_gb: int, color_name: str):
    color_slug = re.sub(r'\s+', '-', color_name.lower().strip())
    sku_id     = f"{model_id}-{storage_gb}gb-{color_slug}"

    tx.run("""
        MATCH (m:Model  {model_id: $model_id})
        MATCH (ms:ModelStorage {model_storage_id: $ms_id})
        MATCH (cv:ColorVariant {color_variant_id: $cv_id})

        MERGE (v:Variant {sku_id: $sku_id})
        ON CREATE SET v.stock = 'in_stock'

        MERGE (m)-[:HAS_VARIANT]->(v)
        MERGE (v)-[:OF_CONFIG]->(ms)
        MERGE (v)-[:HAS_COLOR]->(cv)
    """, {
        'model_id': model_id,
        'ms_id'   : ms_id,
        'cv_id'   : cv_id,
        'sku_id'  : sku_id,
    })


# ─── 5.6 SpecCategory + SpecItems ────────────────────────────

def upsert_spec_category(tx, model_id: str, raw_cat_name: str,
                           items: list):
    """
    items: [{'name': str, 'value': str}, ...]
    Ống kính → SpecItem node riêng
    Còn lại  → features[] property trên SpecCategory
    """
    cat_key = SPEC_CATEGORY_MAP.get(raw_cat_name, raw_cat_name.lower())
    sc_id   = f"{model_id}-{cat_key}"

    # Tách lens specs vs feature specs
    lens_items    = [i for i in items if is_lens_spec(i['name'])]
    feature_items = [i for i in items if not is_lens_spec(i['name'])]

    # Feature list → JSON string để lưu vào property
    features_json = json.dumps(
        [{'name': i['name'], 'value': i['value']} for i in feature_items],
        ensure_ascii=False
    )

    tx.run("""
        MATCH (m:Model {model_id: $model_id})
        MERGE (sc:SpecCategory {spec_category_id: $sc_id})
        ON CREATE SET
            sc.name     = $cat_name,
            sc.features = $features
        ON MATCH SET
            sc.features = $features
        MERGE (m)-[:HAS_SPEC]->(sc)
    """, {
        'model_id': model_id,
        'sc_id'   : sc_id,
        'cat_name': raw_cat_name,
        'features': features_json,
    })

    # Tạo SpecItem node cho từng ống kính
    for item in lens_items:
        item_slug  = re.sub(r'[^a-z0-9]', '-', item['name'].lower())
        si_id      = f"{sc_id}-{item_slug}"
        numeric_val = extract_numeric(item['value'])

        tx.run("""
            MATCH (sc:SpecCategory {spec_category_id: $sc_id})
            MERGE (si:SpecItem {spec_item_id: $si_id})
            ON CREATE SET
                si.name          = $item_name,
                si.value         = $item_value,
                si.numeric_value = $numeric_val
            ON MATCH SET
                si.value         = $item_value,
                si.numeric_value = $numeric_val
            MERGE (sc)-[:HAS_SPEC_ITEM]->(si)
        """, {
            'sc_id'      : sc_id,
            'si_id'      : si_id,
            'item_name'  : item['name'],
            'item_value' : item['value'],
            'numeric_val': numeric_val,
        })


print("✅ Import functions ready")

✅ Import functions ready


## 6. Denormalized Spec Extractor

In [15]:
def extract_denormalized_specs(specs_dict: dict) -> dict:
    """
    Trích các thông số 'đinh' ra thẳng Model node
    để GraphRAG query nhanh mà không cần traverse graph.
    """
    result = {
        'chip'           : None,
        'display_size_in': None,
        'display_hz'     : None,
        'main_camera_mp' : None,
        'weight_g'       : None,
        'ip_rating'      : None,
        'back_material'  : None,
        'frame_material' : None,
        'os'             : None,
    }

    def find_spec(cat_names, item_names):
        """Tìm value trong các category và item names."""
        for cat_name in cat_names:
            cat = specs_dict.get(cat_name, [])
            for item in cat:
                for iname in item_names:
                    if iname.lower() in item['name'].lower():
                        return item['value']
        return None

    # Chip
    chip_val = find_spec(
        ['Vi xử lý & đồ họa', 'Bộ xử lý & Đồ họa'],
        ['Chipset', 'Chip']
    )
    if chip_val:
        # Chuẩn hóa: 'Chip A19 Pro' → 'A19 Pro', 'Apple A18 Pro' → 'A18 Pro'
        chip_clean = re.sub(r'^(Chip |Apple )', '', chip_val).split()[0:3]
        result['chip'] = ' '.join(chip_clean)

    # Display size
    disp_val = find_spec(['Màn hình'], ['Kích thước'])
    if disp_val:
        m = re.search(r'([\d.]+)\s*inches?', disp_val, re.IGNORECASE)
        result['display_size_in'] = float(m.group(1)) if m else None

    # Display Hz
    hz_val = find_spec(['Màn hình'], ['Tần số quét'])
    if hz_val:
        m = re.search(r'(\d+)\s*Hz', hz_val, re.IGNORECASE)
        result['display_hz'] = int(m.group(1)) if m else None

    # Main camera MP
    cam_val = find_spec(['Camera sau'], ['Camera sau', 'Camera chính'])
    if cam_val:
        m = re.search(r'(\d+)\s*MP', cam_val)
        result['main_camera_mp'] = int(m.group(1)) if m else None

    # Weight
    w_val = find_spec(
        ['Kích thước & Trọng lượng', 'Thiết kế & Trọng lượng'],
        ['Trọng lượng']
    )
    if w_val:
        m = re.search(r'(\d+)\s*g(?:ram)?', w_val, re.IGNORECASE)
        result['weight_g'] = int(m.group(1)) if m else None

    # IP Rating
    ip_val = find_spec(
        ['Thông số khác', 'Kích thước & Trọng lượng'],
        ['Kháng nước', 'Chỉ số']
    )
    if ip_val:
        m = re.search(r'IP(\d+)', ip_val)
        result['ip_rating'] = f"IP{m.group(1)}" if m else None

    # Materials
    result['back_material']  = find_spec(
        ['Thiết kế & Trọng lượng'], ['Chất liệu mặt lưng']
    )
    result['frame_material'] = find_spec(
        ['Thiết kế & Trọng lượng'], ['Chất liệu khung']
    )

    # OS
    result['os'] = find_spec(['Tính năng khác'], ['Hệ điều hành'])

    return result


print("✅ Spec extractor ready")

✅ Spec extractor ready


## 7. Main Import Loop

In [16]:
errors   = []
imported = []

with driver.session() as session:
    for idx, row in df.iterrows():
        sku  = row['sku']
        name = row['name']
        print(f"\n[{idx+1}/{len(df)}] Processing: {sku}")

        try:
            # ── Parse model info ──────────────────────────────
            info       = row['_model_info']
            model_id   = sku.split('-')[0] + '-' + '-'.join(sku.split('-')[1:])
            model_id   = sku  # dùng SKU làm model_id
            # Nếu sku có storage suffix (iphone-16-pro-max-512gb),
            # bóc model_id cơ sở
            model_id   = re.sub(r'-\d+(gb|tb)$', '', sku, flags=re.IGNORECASE)

            series_id  = re.sub(r'\s+', '-', info['series_name'].lower())
            storage_gb = info['storage_gb'] or 256  # fallback
            base_price = parse_price(row['base_price'])
            sale_price = parse_price(row['sale_price'])

            # ── Parse specs ───────────────────────────────────
            specs_dict = json.loads(row['specifications'])
            den_specs  = extract_denormalized_specs(specs_dict)

            # ── Parse colors ──────────────────────────────────
            colors_raw = json.loads(row['colors'])
            color_list = list(colors_raw.values())  # [{color, image_url, price}]

            # ── Build color & storage lists for Model node ────
            colors_available = [c['color'] for c in color_list]
            storage_options  = [storage_gb]

            # Build summary for GraphRAG
            summary = (
                f"{info['model_name']} được trang bị chip {den_specs['chip'] or 'N/A'}, "
                f"màn hình {den_specs['display_size_in'] or '?'} inch "
                f"{den_specs['display_hz'] or '?'}Hz, "
                f"camera {den_specs['main_camera_mp'] or '?'}MP, "
                f"khung {den_specs['frame_material'] or '?'}, "
                f"{den_specs['ip_rating'] or '?'}, ra mắt {info['release_year'] or '?'}."
            )

            # ── 1. Brand & Series ─────────────────────────────
            session.execute_write(upsert_brand_series, BRAND_NAME, {
                'series_id'  : series_id,
                'series_name': info['series_name'],
            })

            # ── 2. Model ──────────────────────────────────────
            model_data = {
                'model_id'       : model_id,
                'name'           : info['model_name'],
                'series_id'      : series_id,
                'tier'           : info['tier'],
                'release_year'   : info['release_year'],
                'summary'        : summary,
                'tagline'        : f"Chip {den_specs['chip']}, camera {den_specs['main_camera_mp']}MP, {den_specs['ip_rating']}",
                'colors_available': colors_available,
                'storage_options' : storage_options,
                **den_specs
            }
            session.execute_write(upsert_model, model_data)
            print(f"  ✓ Model: {info['model_name']}")

            # ── 3. ModelStorage ───────────────────────────────
            ms_id = session.execute_write(
                upsert_model_storage,
                model_id, storage_gb, base_price, sale_price
            )
            print(f"  ✓ ModelStorage: {storage_gb}GB @ {sale_price:,.0f}đ")

            # ── 4. ColorVariant + Variant (SKU) ───────────────
            for color_info in color_list:
                color_name  = color_info['color']
                color_price = parse_price(color_info.get('price'))
                image_url   = color_info.get('image_url', '')

                cv_id = session.execute_write(
                    upsert_color_variant,
                    model_id, color_name, image_url, color_price
                )
                session.execute_write(
                    upsert_variant,
                    model_id, ms_id, cv_id, storage_gb, color_name
                )
            print(f"  ✓ {len(color_list)} colors + {len(color_list)} SKUs")

            # ── 5. SpecCategory + SpecItems ───────────────────
            spec_count = 0
            for cat_name, items in specs_dict.items():
                if not items:
                    continue
                session.execute_write(
                    upsert_spec_category,
                    model_id, cat_name, items
                )
                spec_count += 1
            print(f"  ✓ {spec_count} SpecCategories")

            imported.append(sku)

        except Exception as e:
            print(f"  ❌ ERROR: {e}")
            errors.append({'sku': sku, 'error': str(e)})

print(f"\n{'='*50}")
print(f"✅ Done: {len(imported)}/{len(df)} products imported")
if errors:
    print(f"❌ {len(errors)} errors:")
    for e in errors:
        print(f"   {e['sku']}: {e['error']}")


[1/10] Processing: iphone-17-pro-max
  ✓ Model: iPhone 17 Pro Max
  ✓ ModelStorage: 256GB @ 36,990,000đ
  ✓ 3 colors + 3 SKUs
  ✓ 14 SpecCategories

[2/10] Processing: iphone-17-pro
  ✓ Model: iPhone 17 Pro
  ✓ ModelStorage: 256GB @ 34,690,000đ
  ✓ 3 colors + 3 SKUs
  ✓ 14 SpecCategories

[3/10] Processing: iphone-17-256gb
  ✓ Model: iPhone 17
  ✓ ModelStorage: 256GB @ 24,390,000đ
  ✓ 5 colors + 5 SKUs
  ✓ 15 SpecCategories

[4/10] Processing: iphone-16-pro-max
  ✓ Model: Điện thoại iPhone 16 Pro Max
  ✓ ModelStorage: 256GB @ 30,990,000đ
  ✓ 4 colors + 4 SKUs
  ✓ 15 SpecCategories

[5/10] Processing: iphone-17e
  ✓ Model: iPhone 17e
  ✓ ModelStorage: 256GB @ 17,490,000đ
  ✓ 3 colors + 3 SKUs
  ✓ 14 SpecCategories

[6/10] Processing: iphone-15
  ✓ Model: iPhone 15
  ✓ ModelStorage: 128GB @ 17,590,000đ
  ✓ 5 colors + 5 SKUs
  ✓ 15 SpecCategories

[7/10] Processing: iphone-16-pro-max-512gb
  ✓ Model: iPhone 16 Pro Max
  ✓ ModelStorage: 512GB @ 38,490,000đ
  ✓ 4 colors + 4 SKUs
  ✓ 15 Spe

## 8. UPGRADE_OF Relationship (liên kết đời trước-sau)

In [17]:
# Định nghĩa thứ tự upgrade: (model_mới, model_cũ)
UPGRADE_PAIRS = [
    ('iphone-17-pro-max', 'iphone-16-pro-max'),
    ('iphone-17-pro'    , 'iphone-16-pro-max'),  # cross-compare
    ('iphone-16-pro-max', 'iphone-15'),
    ('iphone-17e'       , 'iphone-16e'),
    ('iphone-16e'       , 'iphone-15'),
    ('iphone-17-256gb'  , 'iphone-16e'),
    ('iphone-15'        , 'iphone-14'),
]

with driver.session() as session:
    for newer, older in UPGRADE_PAIRS:
        result = session.run("""
            MATCH (newer:Model {model_id: $newer})
            MATCH (older:Model {model_id: $older})
            MERGE (newer)-[:UPGRADE_OF]->(older)
            RETURN newer.name, older.name
        """, {'newer': newer, 'older': older})
        rec = result.single()
        if rec:
            print(f"  ✓ {rec[0]} → UPGRADE_OF → {rec[1]}")
        else:
            print(f"  ⚠️  Skipped: {newer} or {older} not found")

print("\n✅ UPGRADE_OF relationships created")

  ✓ iPhone 17 Pro Max → UPGRADE_OF → Điện thoại iPhone 16 Pro Max
  ✓ iPhone 17 Pro → UPGRADE_OF → Điện thoại iPhone 16 Pro Max
  ✓ Điện thoại iPhone 16 Pro Max → UPGRADE_OF → iPhone 15
  ✓ iPhone 17e → UPGRADE_OF → iPhone 16e
  ✓ iPhone 16e → UPGRADE_OF → iPhone 15
  ⚠️  Skipped: iphone-17-256gb or iphone-16e not found
  ✓ iPhone 15 → UPGRADE_OF → iPhone 14

✅ UPGRADE_OF relationships created


## 9. SHARES_SPEC Relationship (chip/SpecItem dùng chung)

In [18]:
"""
Nếu 2 model có cùng chip (denormalized field),
tạo SHARES_SPEC relationship thông qua SpecItem.
Dùng cho GraphRAG community detection.
"""

with driver.session() as session:
    result = session.run("""
        MATCH (m1:Model)
        MATCH (m2:Model)
        WHERE m1.model_id < m2.model_id
          AND m1.chip IS NOT NULL
          AND m1.chip = m2.chip
        MERGE (m1)-[:SHARES_CHIP]->(m2)
        RETURN m1.name AS m1, m2.name AS m2, m1.chip AS chip
    """)

    shared = result.data()
    for r in shared:
        print(f"  ✓ {r['m1']} ←SHARES_CHIP→ {r['m2']} ({r['chip']})")

print(f"\n✅ {len(shared)} SHARES_CHIP relationships created")

  ✓ iPhone 17 Pro ←SHARES_CHIP→ iPhone 17 Pro Max (A19 Pro)
  ✓ iPhone 17 ←SHARES_CHIP→ iPhone 17e (A19)

✅ 2 SHARES_CHIP relationships created


## 10. Verify: Đếm Nodes & Relationships

In [19]:
VERIFY_QUERIES = [
    ("Brand"        , "MATCH (n:Brand)         RETURN count(n) AS cnt"),
    ("Series"       , "MATCH (n:Series)        RETURN count(n) AS cnt"),
    ("Model"        , "MATCH (n:Model)         RETURN count(n) AS cnt"),
    ("ModelStorage" , "MATCH (n:ModelStorage)  RETURN count(n) AS cnt"),
    ("ColorFamily"  , "MATCH (n:ColorFamily)   RETURN count(n) AS cnt"),
    ("ColorVariant" , "MATCH (n:ColorVariant)  RETURN count(n) AS cnt"),
    ("Variant(SKU)" , "MATCH (n:Variant)       RETURN count(n) AS cnt"),
    ("SpecCategory" , "MATCH (n:SpecCategory)  RETURN count(n) AS cnt"),
    ("SpecItem"     , "MATCH (n:SpecItem)      RETURN count(n) AS cnt"),
    ("─────────── Relationships ─────────────", None),
    ("HAS_SERIES"   , "MATCH ()-[r:HAS_SERIES]->()     RETURN count(r) AS cnt"),
    ("HAS_MODEL"    , "MATCH ()-[r:HAS_MODEL]->()      RETURN count(r) AS cnt"),
    ("HAS_VARIANT"  , "MATCH ()-[r:HAS_VARIANT]->()    RETURN count(r) AS cnt"),
    ("OF_CONFIG"    , "MATCH ()-[r:OF_CONFIG]->()      RETURN count(r) AS cnt"),
    ("HAS_COLOR"    , "MATCH ()-[r:HAS_COLOR]->()      RETURN count(r) AS cnt"),
    ("HAS_SPEC"     , "MATCH ()-[r:HAS_SPEC]->()       RETURN count(r) AS cnt"),
    ("HAS_SPEC_ITEM", "MATCH ()-[r:HAS_SPEC_ITEM]->()  RETURN count(r) AS cnt"),
    ("UPGRADE_OF"   , "MATCH ()-[r:UPGRADE_OF]->()     RETURN count(r) AS cnt"),
    ("SHARES_CHIP"  , "MATCH ()-[r:SHARES_CHIP]->()    RETURN count(r) AS cnt"),
]

with driver.session() as session:
    print(f"{'Label':<25} {'Count':>8}")
    print("-" * 35)
    for label, q in VERIFY_QUERIES:
        if q is None:
            print(f"\n{label}")
            continue
        cnt = session.run(q).single()['cnt']
        print(f"  {label:<23} {cnt:>8,}")

Label                        Count
-----------------------------------
  Brand                          1
  Series                         5
  Model                          8
  ModelStorage                  10
  ColorFamily                   19
  ColorVariant                  31
  Variant(SKU)                  38
  SpecCategory                 103
  SpecItem                      30

─────────── Relationships ─────────────
  HAS_SERIES                     5
  HAS_MODEL                      9
  HAS_VARIANT                   38
  OF_CONFIG                     38
  HAS_COLOR                     38
  HAS_SPEC                     103
  HAS_SPEC_ITEM                 30
  UPGRADE_OF                     6
  SHARES_CHIP                    2


## 11. Sample Queries cho Chatbot

In [20]:
with driver.session() as session:

    print("=" * 60)
    print("Query 1: Tất cả màu của iPhone 17 Pro Max")
    print("=" * 60)
    result = session.run("""
        MATCH (m:Model {model_id: 'iphone-17-pro-max'})
              -[:HAS_VARIANT]->(v:Variant)
              -[:HAS_COLOR]->(cv:ColorVariant)
              -[:OF_CONFIG]->(ms:ModelStorage)
        WITH m, ms, collect(cv.name) AS colors
        RETURN m.name AS model,
               ms.storage_gb AS storage,
               ms.sale_price AS price,
               colors
        ORDER BY ms.storage_gb
    """)
    for r in result:
        print(f"  {r['model']} {r['storage']}GB | {r['price']:,.0f}đ | {r['colors']}")

    print()
    print("=" * 60)
    print("Query 2: So sánh camera iPhone 17 Pro Max vs 17 Pro")
    print("=" * 60)
    result = session.run("""
        MATCH (m:Model)-[:HAS_SPEC]->(sc:SpecCategory {name: 'Camera sau'})
              -[:HAS_SPEC_ITEM]->(si:SpecItem)
        WHERE m.model_id IN ['iphone-17-pro-max', 'iphone-17-pro']
        RETURN m.name AS model, si.name AS spec, si.value AS value
        ORDER BY m.model_id, si.name
    """)
    for r in result:
        print(f"  [{r['model']}] {r['spec']}: {r['value'][:60]}...")

    print()
    print("=" * 60)
    print("Query 3: Tìm điện thoại có màn hình >= 6.5 inch")
    print("=" * 60)
    result = session.run("""
        MATCH (m:Model)
        WHERE m.display_size_in >= 6.5
        RETURN m.name AS model, m.display_size_in AS screen,
               m.chip AS chip
        ORDER BY m.display_size_in DESC
    """)
    for r in result:
        print(f"  {r['model']} | {r['screen']}" + '" | ' + f"Chip {r['chip']}")

    print()
    print("=" * 60)
    print("Query 4: Tìm model cùng chip A19 Pro")
    print("=" * 60)
    result = session.run("""
        MATCH (m:Model)
        WHERE m.chip CONTAINS 'A19 Pro'
        RETURN m.name AS model, m.tier AS tier
        ORDER BY m.tier
    """)
    for r in result:
        print(f"  {r['model']} ({r['tier']})")

    print()
    print("=" * 60)
    print("Query 5: iPhone nào là upgrade của iPhone 16 Pro Max?")
    print("=" * 60)
    result = session.run("""
        MATCH (newer:Model)-[:UPGRADE_OF]->(older:Model {model_id: 'iphone-16-pro-max'})
        RETURN newer.name AS upgrade_name, newer.chip AS new_chip,
               older.chip AS old_chip
    """)
    for r in result:
        print(f"  {r['upgrade_name']} | Chip: {r['new_chip']} (thay {r['old_chip']})")

Query 1: Tất cả màu của iPhone 17 Pro Max

Query 2: So sánh camera iPhone 17 Pro Max vs 17 Pro
  [iPhone 17 Pro] Camera sau: Chính: 48MP khẩu độ ƒ/1.6 OIS hỗ trợ chụp 24MP hoặc 48MP
Góc...
  [iPhone 17 Pro] Quay video: Quay video 4K Dolby Vision 24/25/30/60/100/120 fps, 1080p 25...
  [iPhone 17 Pro Max] Camera sau: Chính: 48MP khẩu độ ƒ/1.6 OIS hỗ trợ chụp 24MP hoặc 48MP
Góc...
  [iPhone 17 Pro Max] Quay video: Quay video 4K Dolby Vision 24/25/30/60/100/120 fps, 1080p 25...

Query 3: Tìm điện thoại có màn hình >= 6.5 inch
  Điện thoại iPhone 16 Pro Max | 6.9" | Chip A18 Pro
  iPhone 17 Pro Max | 6.9" | Chip A19 Pro

Query 4: Tìm model cùng chip A19 Pro
  iPhone 17 Pro (pro)
  iPhone 17 Pro Max (pro_max)

Query 5: iPhone nào là upgrade của iPhone 16 Pro Max?
  iPhone 17 Pro Max | Chip: A19 Pro (thay A18 Pro)
  iPhone 17 Pro | Chip: A19 Pro (thay A18 Pro)


## 12. GraphRAG: Tạo Text Summary cho Community Detection

In [21]:
"""
GraphRAG cần text content để tạo community summaries.
Cell này lưu thêm 'text_content' vào mỗi Model node —
đây là văn bản mà LLM sẽ đọc khi tạo graph index.
"""

with driver.session() as session:
    # Lấy tất cả model kèm spec categories
    result = session.run("""
        MATCH (m:Model)-[:HAS_SPEC]->(sc:SpecCategory)
        RETURN m.model_id AS mid, m.name AS name, m.summary AS summary,
               collect(sc.name + ': ' + coalesce(sc.features, '')) AS spec_texts
    """)

    for r in result:
        spec_text = '\n'.join(r['spec_texts'][:5])  # giới hạn để tránh quá dài
        full_text = f"{r['name']}\n\n{r['summary']}\n\nThông số:\n{spec_text}"

        session.run("""
            MATCH (m:Model {model_id: $mid})
            SET m.text_content = $text
        """, {'mid': r['mid'], 'text': full_text[:2000]})  # giới hạn 2000 chars

        print(f"  ✓ text_content set for {r['name']}")

print("\n✅ text_content ready for GraphRAG indexing")

  ✓ text_content set for iPhone 17 Pro Max
  ✓ text_content set for iPhone 17 Pro
  ✓ text_content set for iPhone 17
  ✓ text_content set for Điện thoại iPhone 16 Pro Max
  ✓ text_content set for iPhone 17e
  ✓ text_content set for iPhone 15
  ✓ text_content set for iPhone 14
  ✓ text_content set for iPhone 16e

✅ text_content ready for GraphRAG indexing


## 13. Cleanup (optional)

In [22]:
# Chạy cell này NẾU muốn xóa toàn bộ data và làm lại từ đầu
# ⚠️ KHÔNG chạy nếu không cần thiết!

RESET = False  # Đặt True để xóa

if RESET:
    with driver.session() as session:
        session.run("MATCH (n) DETACH DELETE n")
        print("⚠️  All data deleted")
else:
    print("Skip cleanup (RESET = False)")

Skip cleanup (RESET = False)


## 14. Close Connection

In [23]:
driver.close()
print("✅ Neo4j connection closed")
print()
print("📊 Summary:")
print(f"   Products processed : {len(df)}")
print(f"   Successfully imported: {len(imported)}")
print(f"   Errors             : {len(errors)}")

✅ Neo4j connection closed

📊 Summary:
   Products processed : 10
   Successfully imported: 10
   Errors             : 0
